*Six things to check before you spend one of them, with the code, and what each one cost us when we skipped it.*

**You changed your agent. Is it better?** You get five submissions a day, two of them play, and the answer comes back hours later mixed into the noise of whoever you happened to be paired against. So everybody builds something offline to decide what is worth submitting, and then trusts it.

**Ours said 72% on 98 held-out seats. The agent went on to win 8 of its next 60 real ladder games.** Nothing about that test was sloppy: it held out seats, it reported an interval, it was reproducible, and it stayed in service for a week because nothing in it looked broken from the inside. It was wrong by 59 points.

**This notebook is the six checks we added afterwards, and it does not promise you a trustworthy score.** It gives you six specific questions to ask your own harness, each with the code to answer it and the number it cost us to skip it. Four of them are cheap. One of them, check 4, is the one nobody runs, and it is the one that cost us the 72%.

It is short on purpose: a plain question each, a few lines of code, a number. Run them in order, because an early failure makes the later ones meaningless. Nothing here needs any theory, and the argument behind them is a separate notebook you do not have to read.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from pathlib import Path
from scipy import stats


def find(name):
    """Locate an input, wherever this notebook is running.

    EVERY input this notebook uses comes from the attached dataset. Nothing is
    pasted into a cell, so forking it and re-running gives you the same figures
    from the same declared source rather than somebody else's numbers frozen
    into a string literal.
    """
    for root in ("/kaggle/input", ".", "..", "data"):
        r = Path(root)
        if r.exists():
            for p in r.rglob(name):
                return p
    return None


def load_input(name):
    p = find(name)
    if p is None:
        raise FileNotFoundError(
            f"{name} not found. Attach the dataset "
            f"'destbreso/kaggriculture-benchmark-matchups' with Add Input, "
            f"on the right of the editor.")
    return json.loads(Path(p).read_text())

plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 130,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.22, "grid.linewidth": 0.6,
    "font.size": 9.5, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.labelsize": 9.5, "legend.frameon": False,
    # Thin and soft throughout. Heavy strokes and big vertices fight the data.
    "lines.linewidth": 1.1, "lines.markersize": 4.5,
    "lines.solid_capstyle": "round", "lines.dash_capstyle": "round",
    "axes.linewidth": 0.7, "xtick.major.width": 0.7, "ytick.major.width": 0.7,
    "patch.linewidth": 0.7,
})
INK, TEAL, AMBER, GREY, RED = "#0F172A", "#0F766E", "#B45309", "#94A3B8", "#B91C1C"

## The six at a glance

Run them in this order. Each can make the ones after it meaningless, so a failure means stop, not "note it and carry on".

| | the check | the plain question | what skipping it cost us |
|---|---|---|---|
| 1 | **Same game** | are you on the engine the recordings were made on? | banks change by **3x** between engine versions |
| 2 | **Off means off** | with your change disabled, is it byte-identical to before? | a shadowed variable silently rewired the whole farm |
| 3 | **It survives** | does it finish a season without crashing or timing out? | a crash forfeits, and it is silent |
| 4 | **Who you tested against** | how many *different* opponents did you really face? | our 40 opponents were **2** distinct strategies |
| 5 | **Same games, not more games** | did both agents play the identical games? | otherwise you need thousands of games instead of tens |
| 6 | **The number you picked is not the number you get** | did you keep the best of many settings and quote its score? | **20 points**, pure selection, on the very same games |

---

## Check 1. Are you even playing the same game?

Install a different version of the engine and the same seed pays out differently. Not a little differently.

Rayk Kretzschmar's [rank-your-agent notebook](https://www.kaggle.com/code/raykkretzschmar/kaggriculture-rank-your-agent) measured one reference agent banking **28,370 on engine 1.32.3 and 9,002 on 1.32.6**. Factor of three. Same agent, same seed.

And pinning the version then checking the version *string* does not prove you fixed it. So do not check the string. **Replay a game you already know the answer to, and check the money comes back.**

In [ ]:
# The idea, in three lines.
#
#   for m in recorded_games:
#       bank = play(my_agent, opponent=m["their_recorded_actions"], seed=m["seed"])
#       assert abs(bank - m["recorded_bank"]) <= 1
#
# Ours (research/exact_replay.py). Its run record travels with the dataset, so
# the three figures below are counted here rather than typed here: one row per
# recorded matchup, holding what the seat banked and what the replay banked.
R = load_input("exact_replay.json")
err = np.abs([r["replay"] - r["recorded"] for r in R])
exact = sum(1 for r in R if r["exact"])
outcome = sum(1 for r in R if r["recorded_won"] == r["replay_won"])

print(f"  banks reproduced exactly     {exact}/{len(R)}  "
      f"({100 * exact / len(R):.0f} %)")
print(f"  median absolute error        ${np.median(err):,.0f}")
print(f"  win/loss outcome reproduced  {outcome}/{len(R)}  "
      f"({100 * outcome / len(R):.0f} %)")

### Check the behaviour, never the version string

The obvious way to do this check is to assert the engine version, and it does not work. `importlib.metadata.version()` reads the package metadata that was just written to disk, while `import` can still resolve an older copy sitting earlier on `sys.path`. The pin reports success and the engine you are actually running is the old one.

Rayk Kretzschmar's [rank-your-agent notebook](https://www.kaggle.com/code/raykkretzschmar/kaggriculture-rank-your-agent) makes this point and it is worth repeating, because a version assertion is exactly the kind of check that passes forever and protects nothing. **Replay a recorded game and assert the bank comes back.** That is a behavioural fixture, and unlike a string it cannot be satisfied by the wrong code.

---

## Check 2. With your change switched off, is it the same agent?

Put every new behaviour behind a flag. With the flag off, your agent must emit **exactly** the same action as before, on every single turn.

This sounds like paranoia. It is the check that has caught the most real bugs here: a shadowed variable that silently rewired the farm, and a refactor that changed 1,282 of 1,438 turns while looking like a tidy-up.

And do not print "parity OK". Print how many turns you checked.

In [ ]:
# for every turn:   assert my_agent(obs) == old_agent(obs)
#
# Ours (research/instrument.py, gate PARITY). The gate writes down what it
# checked, and that record travels with the dataset, so the count below is read
# back off the run instead of being asserted about it.
G = {g["name"]: g for g in load_input("instrument.json")["gates"]}
turns, diffs = G["PARITY"]["checked"], G["PARITY"]["diffs"]

print(f"  PARITY   {turns:,} turns, {diffs} differences")
print()
print(f"  'parity OK' is a wish. '{turns:,} turns, {diffs} differences' "
      f"is a gate.")

## Check 3. Does it survive a whole season?

An agent that raises an exception gets status `ERROR` and forfeits. And because invalid actions here are **silent no-ops**, an agent can also quietly do nothing for 700 turns and simply lose, with no error anywhere to tell you.

So play it on several seeds against several kinds of opponent, including an idle one, and check three things: it finishes, it emits a well-formed action every turn, and it stays inside the time budget.

Ours: 36 episodes, mean turn time 0.23 ms against a 1,000 ms budget.

---

## Check 4. How many opponents did you actually face?

This is the one nobody checks, and it is the one that cost us the 72%.

We scored candidates against a pool of **40 recorded opponents**. Sounds solid. Then we compared what those 40 opponents actually *build*, tile by tile. **Two farm plans**, under thirty-five names.

Rather than ask you to take that on trust, the cell below measures it on the attached benchmark, which is every replayable matchup in the public episode dataset. Draw a pool of any size you like at random and count how many genuinely different opponents came with it.

In [ ]:
# The UNFILTERED file, deliberately. The other one in this dataset is stratified
# by behaviour, so measuring the collapse on it would measure the fix rather
# than the problem. This is what you get if you simply collect episodes.
BENCH = find("matchups_all.parquet") or find("matchups.parquet")
M = pd.read_parquet(BENCH, columns=["episode_id", "opponent_rating",
                                    "opponent_behaviour",
                                    "recorded_bank_yours",
                                    "recorded_bank_opponent"]).dropna(
                                        subset=["opponent_rating"])
print(f"{len(M):,} matchups, {M.opponent_behaviour.nunique():,} distinct "
      f"behaviours among them")
print(f"{100 * (M.recorded_bank_yours > M.recorded_bank_opponent).mean():.0f} % "
      f"are games the recording side had already won")

In [ ]:
rng = np.random.default_rng(7)

def distinct(beh, k, reps=200):
    """Draw k opponents, count how many genuinely different ones came with them."""
    if k > len(beh):
        return None
    return int(np.median([len(set(rng.choice(beh, k, replace=False)))
                          for _ in range(reps)]))

# Matchmaking is rating-conditioned: it hands you opponents near your own score,
# not a random sample of the field. So the question is not "how much does a pool
# collapse", it is "how much does it collapse WHERE YOU SIT".
BANDS = [(400, 800), (800, 1400), (1400, 2000), (2000, 2400),
         (2400, 2700), (2700, 3200)]
rows = []
for lo, hi in BANDS:
    s = M[(M.opponent_rating >= lo) & (M.opponent_rating < hi)]
    b = s.opponent_behaviour.to_numpy()
    if len(b) < 120:
        continue
    rows.append({"if you sit at": f"{lo}-{hi}", "matchups here": len(b),
                 "draw 40, get": distinct(b, 40),
                 "draw 100, get": distinct(b, 100)})
B = pd.DataFrame(rows)
B["a pool of 40 is really"] = [f"{v} opponents" for v in B["draw 40, get"]]
display(B)

whole = distinct(M.opponent_behaviour.to_numpy(), 40)
print(f"drawing 40 from the whole field instead: {whole} distinct behaviours")

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 3.9))
x = np.arange(len(B))
ax.axhline(40, color=GREY, lw=1.0, ls=(0, (4, 3)))
ax.annotate("what your table claims: 40", xy=(0, 40), xytext=(0, 6),
            textcoords="offset points", fontsize=8.5, color="#475569")
ax.axhline(whole, color=GREY, lw=0.9, ls=(0, (1, 2.5)))
ax.annotate(f"drawn from the whole field: {whole}", xy=(0, whole),
            xytext=(0, 5), textcoords="offset points", fontsize=8.5,
            color="#475569")
ax.plot(x, B["draw 40, get"], color=TEAL, marker="o",
        label="drawn from your own rating band")
for xi, v in zip(x, B["draw 40, get"]):
    ax.annotate(str(v), (xi, v), textcoords="offset points", xytext=(0, 7),
                ha="center", fontsize=9, fontweight="bold", color=INK)
ax.set_xticks(x); ax.set_xticklabels(B["if you sit at"], fontsize=8.5)
ax.set_ylim(0, 45)
ax.set_xlabel("the rating band you are matched into")
ax.set_ylabel("distinct behaviours in a pool of 40")
ax.set_title("A pool of forty is worth less the better you get", loc="left")
ax.legend(loc="lower left", fontsize=8.5)
plt.tight_layout(); plt.show()

**The collapse is not a fixed discount, and it is not about the size of your pool. It is about where you sit.**

Down at the bottom of the ladder a pool of forty is very nearly forty different opponents. Climb, and it falls off a cliff. In the bands where anyone reading this is actually competing, forty recorded opponents are single digits of genuinely different behaviour, and adding more of them buys almost nothing, because the next one you add is the one you already have.

The mechanism is not mysterious once you see it in this order. Weak agents are weak in many different ways. Strong ones are strong in **one** way, because a working idea gets published and copied within days, and unrelated teams then emit identical actions turn for turn, deep into the episode. So the better the neighbourhood matchmaking puts you in, the more your neighbours resemble each other.

That also explains a number we could never account for. Our own pool of forty collapsed to **19 distinct behaviours and two farm plans**, and we had no story for why it was so much worse than we expected. This is the story: we did not draw forty opponents from the field, we drew forty from our own neighbourhood.

So the honest denominator for any win rate you quote is the solid line at *your* band, not the number of rows in your table. "Sixteen of sixteen against the pool" is a much smaller claim than it sounds, and how much smaller is now something you can read off a chart rather than guess at.

One more property, easier to fix than any of this: a pool made of games your previous agent **won** cannot show you a loss, and losing is the region a new candidate most needs testing in. Ours was 34 of 40. The file above is drawn without regard to who won, which is why its own figure sits near half, and that is worth demanding of any pool you build.

### And a control, while you have the pool open

Every check in this notebook asks whether your **agent** is sound. This one asks whether your **data** is, and it is the only thing here that can fail while your agent is perfect.

Both seats in this game have the same action set, the same endowment and the same payoff rule. So over episodes whose seat assignment nobody chose, the recorded margin has to look the same from either side. Exactly, with no distributional assumption. A seat-dependent fault anywhere in the chain, in the recording, in your replay, in your extraction, or in an agent that reads a field seat 1 does not receive, shows up here and is invisible to everything else.

In [ ]:
from scipy import stats
E = (M.assign(_r=np.random.default_rng(17).random(len(M)))
      .sort_values("_r").drop_duplicates("episode_id"))
mg = (E.recorded_bank_yours - E.recorded_bank_opponent).to_numpy()

# Exact ties are excluded, because a sign test is defined on the non-zero
# differences and this game produces real ties: two agents running the same code
# bank the same money to the dollar. Counting them as losses makes a sound
# corpus look asymmetric, at p < 0.001 on this data.
nz = mg[mg != 0]
print(f"  episodes                 {len(mg):>10,}")
print(f"  exact ties               {int((mg == 0).sum()):>10,}")
print(f"  win rate, ties excluded  {(nz > 0).mean():>10.4f}   (0.5 if sound)")
print(f"  sign test p              "
      f"{stats.binomtest(int((nz > 0).sum()), len(nz), 0.5).pvalue:>10.3f}")
print(f"  KS against its mirror    {stats.ks_2samp(mg, -mg).pvalue:>10.3f}")

A control is a quantity whose value you know in advance if nothing is wrong, and that is what makes it different from every other number here: it cannot flatter you. If it fails, stop and fix the data, because nothing measured on top of it means anything.

*The derivation, the exact-tie trap that makes this fail on a sound corpus, and the rest of the base distributions of this game are in [Kaggriculture Know Your Noise](https://www.kaggle.com/code/destbreso/kaggriculture-know-your-noise).*

*Why forty opponents can be two: they are related. Unrelated teams here emit identical actions turn for turn, deep into the episode, which we measured in [everyone is playing the same opening](https://www.kaggle.com/code/destbreso/everyone-is-playing-the-same-opening). What you meet also depends on where you sit, measured in [the leaderboard is a habitat gradient](https://www.kaggle.com/code/destbreso/the-leaderboard-is-a-habitat-gradient).*

## Check 5. Play the identical games, not just more games

The cheapest real improvement in this whole notebook. When you compare two agents, make them play **the same seed, from the same seat, against the same opponent**, then compare game by game and take the difference.

Not "run 200 games each and compare the averages". The same games.

Most of the variation in a Kaggriculture bank is not you. It is the seed, the shop draw, the weeds, and who you were paired with. Play the identical game twice and all of that cancels, leaving the part that is actually your change.

In [ ]:
# WRONG: two independent samples, and most of the noise survives.
#   mine  = [play(new) for _ in range(200)]
#   yours = [play(old) for _ in range(200)]
#   compare(mean(mine), mean(yours))
#
# RIGHT: the same games, differenced.
#   for g in games:
#       d = play(new, seed=g.seed, seat=g.seat, opp=g.opponent) \
#         - play(old, seed=g.seed, seat=g.seat, opp=g.opponent)
#
# then a sign test on the differences, not a t-test on the levels.
print("Ours: 40 recorded matchups, each replayed at its own seed, from its own")
print("seat, against that opponent's own recorded actions.")

### How many games? Work backwards, not forwards

The pool size is the one number everybody inherits and nobody derives. Ours was **40** for a week, and the honest reason is that the script which built the first pool has forty as its argparse default. Nobody chose it, and it turns out to be a bad number.

Here is what forty buys, and the second column is the one that hurts.

In [ ]:
def sign_power(n, p, alpha=0.05):
    """Exact power of a two-sided paired sign test against a true win rate p.

    Under the null the wins are Binomial(n, 1/2) and the test rejects outside
    the central 1-alpha region, so the power is the mass a Binomial(n, p) puts
    outside that same region. No normal approximation: at these counts it
    flatters the design by several points.
    """
    lo = stats.binom.ppf(alpha / 2, n, 0.5) - 1
    hi = stats.binom.ppf(1 - alpha / 2, n, 0.5) + 1
    return float(stats.binom.cdf(lo, n, p) + stats.binom.sf(hi - 1, n, p))

def n_for(p, target=0.80):
    return next((n for n in range(10, 2000) if sign_power(n, p) >= target), None)

SECONDS_PER_GAME = 1.13
rows = []
for p in (0.55, 0.575, 0.60, 0.65, 0.70):
    n = n_for(p)
    rows.append({"to detect a real": f"{100*p:.1f} %",
                 "you need": n,
                 "costing": f"{2*n*SECONDS_PER_GAME/60:.0f} min",
                 "power at n=40": f"{100*sign_power(40, p):.0f} %",
                 "power at n=421": f"{100*sign_power(421, p):.0f} %"})
display(pd.DataFrame(rows))

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 4.2))
ns = np.unique(np.round(np.logspace(np.log10(20), np.log10(1200), 34)).astype(int))
for p, col in ((0.70, GREY), (0.65, AMBER), (0.60, TEAL), (0.55, RED)):
    ax.plot(ns, [sign_power(int(n), p) for n in ns], color=col,
            label=f"a real {100*p:.0f} %")
ax.axhline(0.80, color=INK, lw=0.8, ls=(0, (4, 3)))
ax.annotate("80 % power", xy=(21, 0.80), xytext=(0, 5),
            textcoords="offset points", fontsize=8.5, color=INK)
for n, lab in ((40, "the 40\nwe inherited"), (421, "the 421\nin this dataset")):
    ax.axvline(n, color=GREY, lw=0.7)
    ax.annotate(lab, xy=(n, 0.06), xytext=(4, 0), textcoords="offset points",
                fontsize=8.5, color="#475569", va="bottom")
ax.set_xscale("log"); ax.set_ylim(0, 1.02)
ax.set_xlabel("paired games"); ax.set_ylabel("chance of noticing a real effect")
ax.set_title("What your pool size can and cannot see", loc="left")
ax.legend(loc="lower right", fontsize=8.5, title="effect size")
plt.tight_layout(); plt.show()

**At forty games, a genuinely better agent that wins 60% of the paired comparisons is noticed 21% of the time.** Four times in five it comes back as "no difference", and "no difference" is not a neutral answer: it is the answer that makes you throw the change away.

So the procedure is three steps, and none of them is picking a sample size.

**One. Decide the smallest effect you would act on.** This is a judgement about your competition, not about statistics, and you can anchor it with a number you already have. Check 6 below measures the winner's curse at **20 points** of win rate, and the screen-to-confirm drop at 8. So a candidate that screens at 65% confirms near 57%, and anything worth acting on lives between 57% and 60%.

**Two. Read the games off the curve.** Detecting a real 60% takes 199 games, 57.5% takes 352, and 55% takes 786. Notice how fast that grows: halving the effect roughly quadruples the games, which is the usual square-law and the reason small effects are expensive to prove.

**Three. Check what it costs.** A replayed game here takes about a second, so the entire useful range is between two and forty minutes. **Cost is not the constraint, and it never was.** Forty was not a trade-off anyone made; it was a default nobody looked at.

That last point generalises past this competition. When a sample size in your work is round, inherited, and equal to a library default, it is worth ten minutes to ask what it can see.

---

## Check 6. If you tried fifty settings and kept the best, that score is a lie

We searched eight parameters over 259 configurations and kept the winner. On the half of the games it searched, it won **14 of 20**. On the other half, which it had never been evaluated against once, it won **10 of 20**.

The agent did not change between those two numbers. **Twenty points of the first one was the act of choosing it.**

Both figures are that one agent on one pool of forty recorded games split in half: 20 games the search scored its configurations against, and 20 it never saw.

The fix costs nothing: **split your games in half, tune on one half, report the other.**

Rayk's notebook has the cleanest example of this I have seen anywhere. He built an all-cow variant of one of his reference agents, measured it **+4,450** better on the seeds he tuned it on, then ran it on held-out seeds where it **lost 3 to 9 by −3,627**. Same agent, same opponent, different seeds. He killed the idea and published the autopsy, which is a better result than a new tier would have been.

## Running all six

One command, the same checks in the same order, every time.

```
python research/instrument.py --agent agents/MINE/main.py \
                              --baseline agents/PREVIOUS/main.py
```

**What follows is the LAYOUT of what comes back, not a transcript of one run.** The figures beside each line are the kind of thing it carries, from runs of mine against a pool of 40 recorded matchups; yours will differ, and the arrangement is the part worth copying. The parity line is the one computed above, in check 2.

```
  GATES
    pass  ENGINE      6/6 banks reproduced, median error $0
    pass  PARITY      4,314 turns, 0 differences
    pass  SAFETY      SAFE TO SUBMIT (on this evidence).
    pass  POPULATION  nominal 40, EFFECTIVE 17, 4 already won, seats 18/22
    pass  SYMMETRY    seats 18/22, p = 0.674

  SAMPLE AUDIT, what each check needs and what it got
    POPULATION   17    5  ok    covers 68 % of the field it was drawn from,
                                at least 84 behaviours unseen
    SYMMETRY     18    8  ok    seats 18/22, p = 0.674
    PAIRED       40  200  THIN  smallest detectable win rate at 80 % power: 72 %
                                power against a real 60 %: 21 %, against 57.5 %: 13 %
    RANKING       2   20  THIN  below this the transitivity check cannot fire

  Phi(mu/sigma): NOT AVAILABLE. This pool plays each opponent once, so there is
  no within-pairing sigma to estimate. Use the win rate and its Wilson interval.

  VERDICT: ...
```

Four things in that output are worth copying even if you build your own harness.

**A coverage figure, not just a count.** "Effective 17" says how many independent tests you have. "Covers 68 % of the field, at least 84 behaviours unseen" says whether they are the tests that matter, and it comes free from counting how many behaviours you saw exactly once. Those are different failures with different fixes: a redundant pool is fixed by adding variety, an incomplete one is not fixed by replaying what you already have.

**A refusal.** The line about `Phi(mu/sigma)` is the harness declining to give a number. That rule turns a margin into a win probability and it is well calibrated, but it is defined on the margin against **one** opponent across seeds. A pool that plays forty different opponents once each has no within-pairing spread to estimate, so the figure it would print is the spread *between* opponents wearing the same symbol, and dividing by it charges your agent for the variance of who it drew. Printing nothing is the correct output. *(Why this is the right call, and what shape the margin actually has, is in [Know Your Noise](https://www.kaggle.com/code/destbreso/kaggriculture-know-your-noise).)*

**The sample audit.** Every check declares what sample it needs, and the run says which ones did not get it. The declaration is the important half: a threshold you can point at is a threshold somebody can argue with, and the `PAIRED` row above is failing against its own, which is the harness telling you the pool is too small before it tells you anything about the agent.

Read that row carefully, because it is the difference between *no effect* and *not enough games*, and people who have not checked write those two down the same way. The 72% is about the **paired** comparison, meaning how often your new agent beats your old one on the same game, not how often it beats the field. And the number beside it is the one that decides things: a change that genuinely wins 60% of paired comparisons is noticed **21%** of the time at this sample size.

**A verdict with its limits printed underneath.** Every opponent in this harness is a recording. It replays its actions and cannot react to yours. So this measures *did my change break the agent, and does it move the margin against the field I actually met*. It does not measure *will this rate higher*, and no number of extra games turns one into the other.

---

## And when a check tells you something you would rather not believe

Then you need to know what it is actually measuring, what it cannot see, and how sure anybody is of any of it. That is a longer argument than a tool should carry, and it lives in **[Kaggriculture Measure Your Agent](https://www.kaggle.com/code/destbreso/kaggriculture-measure-your-agent)**: the harness these six checks are the front of, the four ways a measurement lies, what you are actually maximising, and the tests that decide whether the harness itself works.

Nothing there changes what any check above does. You can stop here and lose nothing you would use.